# Classification metrics with ThreeWToolkit

This notebook walks through **loading the 3W dataset**, **building a preprocessing and feature pipeline**, **training an MLP**, and **evaluating predictions** with the metrics exposed in `ThreeWToolkit.metrics`.


## What you will do

This notebook is divided into steps (data → split → features → train → evaluate), that encompases: 

1. **Load** parquet events and pick a sensor column and target classes.
2. **Split** the dataset into train, validation, and test subsets (random indices).
3. **Fit** `TransformConfig` pipeline on the full dataset, then **transform** each split.
4. **Train** on the training / validation sets and **predict** on the test set.
5. **Compute metrics**: accuracy, balanced accuracy, AP, precision, recall, F1, and ROC AUC.


### Imports

In [1]:
from ThreeWToolkit.preprocessing import (
    CleanSignalsConfig,
    FillLabelsConfig,
    ImputeMissingConfig,
    NormalizeConfig,
    RemapClassConfig,
    SequentialPreprocessingAdapterConfig,
)
from ThreeWToolkit.trainer import TorchTrainerConfig
from numpy.random import shuffle
from ThreeWToolkit.feature_extraction import (
    ConcatFeatureAdapterConfig,
    SequentialFeatureAdapterConfig,
    StatisticalConfig,
    WindowingConfig,
)
from ThreeWToolkit.models.mlp import MLPConfig
from ThreeWToolkit.dataset import ParquetDatasetConfig, TransformConfig, SubsetDataset
from ThreeWToolkit.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

import numpy as np
import torch
from torch import nn


Let's tell Python to **not print** warnings that match the filter (here, modules under `sklearn.metrics`). This only affects **console output**; it does **not** change numeric metric results.

In [2]:
import warnings
warnings.filterwarnings(
  "ignore",
  module=r"sklearn\.metrics.*",
)

---

## 1. Load the dataset
Let's load the dataset with **`ParquetDataset`** class.


In [3]:
dataset_path = "../../dataset"
selected_col = "T-TPT"

ds_config = ParquetDatasetConfig(
    path=dataset_path,
    clean_data=True,
    target_class=[0, 1, 2],
    columns=[selected_col],
    event_type=["real"],
)
ds = ds_config.build()
ds[19]


2026-04-16 21:00:53,328 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset found at ../../dataset
2026-04-16 21:00:53,329 | INFO | ThreeWToolkit.dataset.parquet_dataset | Validating dataset integrity...
2026-04-16 21:00:53,329 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset integrity check passed!


DatasetOutputs(signal=                        T-TPT
timestamp                    
2017-04-24 22:01:56  116.5984
2017-04-24 22:01:57  116.5984
2017-04-24 22:01:58  116.5984
2017-04-24 22:01:59  116.5984
2017-04-24 22:02:00  116.5984
...                       ...
2017-04-25 03:59:56  116.4890
2017-04-25 03:59:57  116.4890
2017-04-25 03:59:58  116.4889
2017-04-25 03:59:59  116.4888
2017-04-25 04:00:00  116.4887

[21485 rows x 1 columns], label=timestamp
2017-04-24 22:01:56    <NA>
2017-04-24 22:01:57    <NA>
2017-04-24 22:01:58    <NA>
2017-04-24 22:01:59    <NA>
2017-04-24 22:02:00    <NA>
                       ... 
2017-04-25 03:59:56       0
2017-04-25 03:59:57       0
2017-04-25 03:59:58       0
2017-04-25 03:59:59       0
2017-04-25 04:00:00       0
Name: class, Length: 21485, dtype: Int16, metadata={'file_name': PosixPath('0/WELL-00001_20170424220156.parquet'), 'event_class': 0, 'event_type': 'real'})

---

## 2. Train / validation / test split

Let's apply the split into train (80%), validation (10%), and test (10%) using `SubsetDataset` (views over the same underlying data). 

Note that first we shuffle **indices** (not rows in memory) and build `SubsetDataset` views.


In [4]:
def split_dataset(
    dataset, size_training: float = 0.8, size_validation: float = 0.2
) -> tuple[SubsetDataset, SubsetDataset]:
    if size_training + size_validation != 1:
        raise ValueError("The sum of the sizes must be 1")

    len_dataset = len(dataset)
    lst_indices = list(np.arange(len_dataset))
    shuffle(lst_indices)

    training_indices = lst_indices[: int(len_dataset * size_training)]
    validation_indices = lst_indices[int(len_dataset * size_training) :]
    training_set = SubsetDataset(dataset, indices=training_indices)
    validation_set = SubsetDataset(dataset, indices=validation_indices)
    return training_set, validation_set


ds_train, ds_val_test = split_dataset(ds, size_training=0.8, size_validation=0.2)
ds_val, ds_test = split_dataset(ds_val_test, size_training=0.5, size_validation=0.5)


---

## 3. Preprocessing and features

We will use the **`TransformConfig` pipeline** to apply: 

- **Preprocessing:** clean, impute, normalize, fill labels, remap classes.
- **Features:** sliding windows (`WindowingConfig`) then concatenated statistical descriptors (`StatisticalConfig` via `ConcatFeatureAdapterConfig`).

**Important:** `fit(ds)` uses the **full** dataset so normalization and any statistics are **global**.

Then `transform` each split to get tensor-ready datasets for the MLP.


In [5]:
window_size = 1000
dataset_processor = TransformConfig(
    pre_processing=SequentialPreprocessingAdapterConfig(
        steps=[
            CleanSignalsConfig(missing_column_threshold=1.1),
            ImputeMissingConfig(),
            NormalizeConfig(),
            FillLabelsConfig(),
            RemapClassConfig(),
        ]
    ),
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(window_size=window_size),
            ConcatFeatureAdapterConfig(steps=[StatisticalConfig()]),
        ]
    ),
).build()

dataset_processor.fit(ds)
ds_train_transformed = dataset_processor.transform(ds_train)
ds_val_transformed = dataset_processor.transform(ds_val)
ds_test_transformed = dataset_processor.transform(ds_test)


---

## 4. Model and trainer

Let's define an **MLP** classifier (`MLPConfig`). We use `build()` to create a **trainer** object (optimizer, loss, epochs, batches). Weights are initialized when training begins.

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mlp_config = MLPConfig(
    hidden_sizes=(32, 16),
    activation_function=nn.ReLU(),
    output_size=dataset_processor.num_classes,
)

trainer = TorchTrainerConfig(
    config_model=mlp_config,
    learning_rate=0.001,
    epochs=100,
    batch_size=64,
    device=device,
).build()


2026-04-16 21:01:15,376 | INFO | ThreeWToolkit.core.base_trainer | Initialized TorchTrainer with seed=42
2026-04-16 21:01:15,377 | INFO | ThreeWToolkit.trainer.torch_trainer | TorchTrainer initialized | device=cuda | epochs=100 | batch_size=64


---

## 5. Training

In [7]:
train_results = trainer.train(
    train_dataset=ds_train_transformed, val_dataset=ds_val_transformed
)


2026-04-16 21:01:15,389 | INFO | ThreeWToolkit.core.base_trainer | Starting training | train_size=496 | val_size=62
2026-04-16 21:01:15,428 | INFO | ThreeWToolkit.core.base_trainer | Dataset validation passed
2026-04-16 21:01:15,430 | INFO | ThreeWToolkit.core.base_trainer | Preparing training data...
2026-04-16 21:01:15,430 | INFO | ThreeWToolkit.trainer.torch_trainer | Converting dataset to DataLoader (size=496)
2026-04-16 21:01:21,491 | INFO | ThreeWToolkit.trainer.torch_trainer | Created DataLoader | batches=164
2026-04-16 21:01:21,493 | INFO | ThreeWToolkit.core.base_trainer | Preparing validation data...
2026-04-16 21:01:21,493 | INFO | ThreeWToolkit.trainer.torch_trainer | Converting dataset to DataLoader (size=62)
2026-04-16 21:01:22,243 | INFO | ThreeWToolkit.trainer.torch_trainer | Created DataLoader | batches=21
2026-04-16 21:01:22,244 | INFO | ThreeWToolkit.core.base_trainer | Initializing training state...
2026-04-16 21:01:22,244 | INFO | ThreeWToolkit.trainer.torch_traine

Training:   0%|          | 0/100 [00:00<?, ?epoch/s]

2026-04-16 21:01:41,332 | INFO | ThreeWToolkit.core.base_trainer | Training completed successfully


---

## 6. Predictions on the test set

Inference on held-out data. `trainer.predict` applies the model and, for multi-class, **`argmax` on logits** → **integer class labels** `y_pred`. `trainer.predict_proba` applies **softmax** to logits → **class probabilities** `y_proba` (rows sum to 1), i.e. the **`y_score`** input expected by AP and ROC AUC.

Metrics based on **confusion counts** (accuracy, precision, recall, F1, balanced accuracy) can use **hard** `y_pred`. **Average precision** and **ROC AUC** require **continuous scores**—use **`y_proba`**, not argmax labels.


In [8]:
test_results = trainer.predict(ds_test_transformed)
y_true = test_results.y_true
y_pred = test_results.y_pred

# We call `predict_proba` to get the softmax probabilities
y_proba = trainer.predict_proba(ds_test_transformed).y_pred

2026-04-16 21:01:41,355 | INFO | ThreeWToolkit.trainer.torch_trainer | Converting dataset to DataLoader (size=62)
2026-04-16 21:01:42,134 | INFO | ThreeWToolkit.trainer.torch_trainer | Created DataLoader | batches=20
2026-04-16 21:01:42,144 | INFO | ThreeWToolkit.trainer.torch_trainer | Converting dataset to DataLoader (size=62)
2026-04-16 21:01:42,904 | INFO | ThreeWToolkit.trainer.torch_trainer | Created DataLoader | batches=20


---

## 7. Example sample weights

We use a vector called sample_weight, with length n_samples, to assign a weight to each sample when computing weighted metrics. Each sample contributes proportionally to its assigned weight.

In this example, samples belonging to true class 0 are given a higher importance by assigning them a weight of 2, while all other samples keep the default weight of 1.

Many metrics support sample_weight, which allows you to emphasize specific samples. This is especially useful for handling imbalanced datasets or giving more importance to rare classes.

Here, all samples start with a weight of 1, and only those whose true label is class 0 are increased to weight 2.

In [9]:
sample_weight = np.ones(len(y_true))
sample_weight[y_true == 0] = 2

**Random weights (for demonstration only)**

We can also assign random weights in the range `(0, 1)` to each sample to demonstrate how the `sample_weight` works. This approach is only for illustration purposes and should not be used as a proper evaluation method in real experiments.

In [10]:
sample_weight_random = np.random.rand(len(y_true))

---

## 8. Accuracy (`accuracy_score`)

**What it is:** It is the **fraction of correct predictions** among all samples when `normalize=True` (default); when `normalize=False`, it returns the **count** of correct predictions. 

Weighted accuracy uses `sample_weight` to change each sample’s contribution to that fraction.


In [11]:
acc = accuracy_score(y_true=y_true, y_pred=y_pred)
acc_weighted = accuracy_score(
    y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
)
acc, acc_weighted


(0.971830985915493, 0.9857142857142858)

---

## 9. Balanced accuracy (`balanced_accuracy_score`)

**What it is:** It is the **average of recall obtained on each class**. For multiclass, that is the **unweighted mean of per-class recall** (true positives divided by support for that class). It equals ordinary accuracy when classes are balanced; when they are not, it is typically **less optimistic** than raw accuracy if one class dominates.


In [12]:
balanced_acc = balanced_accuracy_score(y_true=y_true, y_pred=y_pred)
balanced_acc_weighted = balanced_accuracy_score(
    y_true=y_true, y_pred=y_pred, sample_weight=sample_weight_random
)
balanced_acc, balanced_acc_weighted


(0.3333333333333333, 0.3333333333333333)

---

## 10. Average precision (`average_precision_score`)

**What it is:** It summarizes a **precision–recall curve**: the weighted mean of precisions at successive decision thresholds, where the weight is the **increase in recall** from the previous threshold. 

In [13]:
ap = average_precision_score(y_true=y_true, y_pred=y_proba, average="weighted", num_classes=dataset_processor.num_classes)
ap_weighted = average_precision_score(
    y_true=y_true, y_pred=y_proba, sample_weight=sample_weight, num_classes=dataset_processor.num_classes
)
ap, ap_weighted


(0.9691906509118654, 0.21490165670010083)

---

## 11. Precision (`precision_score`)

**What it is:** It is **TP / (TP + FP)** for a class (multiclass uses **one-vs-rest** per label unless otherwise specified). It answers: *among all predictions of class k, what fraction were correct?*

In [14]:
# Multi-class: set `average` explicitly (sklearn default `binary` is only for two classes).
# - `None`: one precision per class (one-vs-rest)
# - `macro`: unweighted mean of per-class precisions
# - `micro`: global TP / (TP + FP) pooled over all classes and samples
# - `weighted`: mean of per-class precisions weighted by support
precision_per_class = precision_score(y_true=y_true, y_pred=y_pred, average=None, pos_label=dataset_processor.num_classes)
for i, p in enumerate(precision_per_class):
    print(f"Precision for class {i}: {(p * 100):.3f}%")

precision_macro_with_weight = precision_score(
    y_true=y_true, y_pred=y_pred, sample_weight=sample_weight, average="macro"
)
print(f"macro (with sample_weight): {(precision_macro_with_weight * 100):.3f}%")

precision_macro = precision_score(y_true=y_true, y_pred=y_pred, average="macro")
precision_micro = precision_score(y_true=y_true, y_pred=y_pred, average="micro")
precision_weighted_avg = precision_score(
    y_true=y_true, y_pred=y_pred, average="weighted"
)
print(f"macro: {(precision_macro * 100):.3f}%")
print(f"micro: {(precision_micro * 100):.3f}%")
print(f"weighted: {(precision_weighted_avg * 100):.3f}%")


Precision for class 0: 97.183%
Precision for class 1: 0.000%
Precision for class 2: 0.000%
macro (with sample_weight): 32.857%
macro: 32.394%
micro: 97.183%
weighted: 94.446%


---

## 12. Recall (`recall_score`)

**What it is:** (sensitivity, true positive rate) is **TP / (TP + FN)** for each class: *among all true instances of class k, what fraction did we retrieve?*


In [15]:
recall_per_class = recall_score(y_true=y_true, y_pred=y_pred, average=None)
for i, p in enumerate(recall_per_class):
    print(f"Recall for class {i}: {(p * 100):.3f}%")

recall_macro_sw = recall_score(
    y_true=y_true, y_pred=y_pred, sample_weight=sample_weight, average="macro"
)
print(f"macro (with sample_weight): {(recall_macro_sw * 100):.3f}%")

recall_macro = recall_score(y_true=y_true, y_pred=y_pred, average="macro")
recall_micro = recall_score(y_true=y_true, y_pred=y_pred, average="micro")
recall_weighted_avg = recall_score(y_true=y_true, y_pred=y_pred, average="weighted")
print(f"macro: {(recall_macro * 100):.3f}%")
print(f"micro: {(recall_micro * 100):.3f}%")
print(f"weighted: {(recall_weighted_avg * 100):.3f}%")


Recall for class 0: 100.000%
Recall for class 1: 0.000%
Recall for class 2: 0.000%
macro (with sample_weight): 33.333%
macro: 33.333%
micro: 97.183%
weighted: 97.183%


---

## 13. F1 score (`f1_score`)

**What it is:** The **F1 score** is the **harmonic mean** of precision and recall: F1 = 2PR / (P + R) (per class or aggregated via **`average`**). 

In [16]:
f1_per_class = f1_score(y_true=y_true, y_pred=y_pred, average=None, sample_weight=sample_weight)
for i, p in enumerate(f1_per_class):
    print(f"F1 for class {i}: {(p * 100):.3f}%")

f1_macro = f1_score(y_true=y_true, y_pred=y_pred, average="macro")
f1_micro = f1_score(y_true=y_true, y_pred=y_pred, average="micro")
f1_weighted = f1_score(y_true=y_true, y_pred=y_pred, average="weighted")
print(f"macro: {(f1_macro * 100):.3f}%")
print(f"micro: {(f1_micro * 100):.3f}%")
print(f"weighted: {(f1_weighted * 100):.3f}%")

f1_weighted_sw = f1_score(
    y_true=y_true, y_pred=y_pred, average="weighted", sample_weight=sample_weight
)
print(f"weighted (with sample_weight): {(f1_weighted_sw * 100):.3f}%")


F1 for class 0: 99.281%
F1 for class 1: 0.000%
F1 for class 2: 0.000%
macro: 32.857%
micro: 97.183%
weighted: 95.795%
weighted (with sample_weight): 97.862%


---

## 14. ROC AUC (`roc_auc_score`)

**What it is:** It is the **area under the receiver operating characteristic curve**: **true positive rate vs. false positive rate** as the classification threshold varies. 

It requires **continuous scores** (`y_score`), so we pass **`y_proba`**.

In [17]:
roc_auc_per_class = roc_auc_score(
    y_true=y_true, y_pred=y_proba, average=None, multi_class="ovr", num_classes=dataset_processor.num_classes
)
for i, p in enumerate(roc_auc_per_class):
    print(f"ROC AUC for class {i}: {(p * 100):.3f}%")

roc_auc_sw = roc_auc_score(
    y_true=y_true, y_pred=y_proba, sample_weight=sample_weight, multi_class="ovr", num_classes=dataset_processor.num_classes
)
print(f"OVR (with sample_weight): {(roc_auc_sw * 100):.3f}%")

print("\nAggregated multi-class ROC AUC:")
roc_auc_ovr_macro = roc_auc_score(
    y_true=y_true, y_pred=y_proba, multi_class="ovr", average="macro", num_classes=dataset_processor.num_classes
)
roc_auc_ovr_weighted = roc_auc_score(
    y_true=y_true, y_pred=y_proba, multi_class="ovr", average="weighted", num_classes=dataset_processor.num_classes
)
roc_auc_ovo_macro = roc_auc_score(
    y_true=y_true, y_pred=y_proba, multi_class="ovo", average="macro", num_classes=dataset_processor.num_classes
)
roc_auc_ovo_weighted = roc_auc_score(
    y_true=y_true, y_pred=y_proba, multi_class="ovo", average="weighted", num_classes=dataset_processor.num_classes
)
print(f"OVR macro: {(roc_auc_ovr_macro * 100):.3f}%")
print(f"OVR weighted: {(roc_auc_ovr_weighted * 100):.3f}%")
print(f"OVO macro: {(roc_auc_ovo_macro * 100):.3f}%")
print(f"OVO weighted: {(roc_auc_ovo_weighted * 100):.3f}%")


ROC AUC for class 0: 87.762%
ROC AUC for class 1: nan%
ROC AUC for class 2: 90.345%
ROC AUC for class 3: nan%
ROC AUC for class 4: 91.252%
OVR (with sample_weight): nan%

Aggregated multi-class ROC AUC:
OVR macro: nan%
OVR weighted: 87.857%
OVO macro: nan%
OVO weighted: 87.857%
